In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

features_cols = [
    "is_new_device",
    "is_new_ip_country", 
    "is_new_withdrawal_address",
    "session_to_withdrawal_secs",
    "hour_deviation",
    "kyc_tier_enc",
    "account_age_days",
]

df = pd.read_csv("data/ato_sessions.csv")
X = df[features_cols]
y = df["is_fraud"]

# Temporal split — correct way to evaluate fraud models
df["timestamp"] = pd.to_datetime(df["timestamp"])
split_date = df["timestamp"].quantile(0.8)
train_idx = df["timestamp"] <= split_date
test_idx  = df["timestamp"] > split_date

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Train
model = XGBClassifier(scale_pos_weight=15, random_state=42) # 15:1 ratio of fraud
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

joblib.dump(model, "models/ato_predict_xgb.pkl")
print("Model saved.")